In [1]:
!pip install pystac


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [1]:
from sedona.spark import SedonaContext
import os
from time import time
from sedona.spark import dataframe_to_arrow
from sedona.spark.geoarrow.geoarrow import create_spatial_dataframe

1.8.0


## GeoPandas DataFrame from Sedona Spatial DataFrame

In [7]:
%%capture
bucket_name = os.environ.get("SEDONA_SOURCE_BUCKET", "apache-sedona-book")

config = SedonaContext.builder()

sedona = SedonaContext.create(config.getOrCreate())

sedona.sparkContext.setLogLevel("ERROR")

sc = sedona.sparkContext

In [8]:
sedona.read.format("geoparquet").\
    load(f"s3a://{bucket_name}/source_data/transportation_barcelona/barcelona.geoparquet").columns

['id',
 'geometry',
 'bbox',
 'version',
 'sources',
 'subtype',
 'class',
 'names',
 'connectors',
 'routes',
 'subclass',
 'subclass_rules',
 'access_restrictions',
 'level_rules',
 'destinations',
 'prohibited_transitions',
 'road_surface',
 'road_flags',
 'speed_limits',
 'width_rules']

In [9]:
df = sedona.read.format("geoparquet").\
    load(f"s3a://{bucket_name}/source_data/transportation_barcelona/barcelona.geoparquet").\
    select("id", "bbox", "version", "sources", "geometry", "subtype", "class", "names", "connectors", "routes", "subclass", "level_rules", "road_surface", "road_flags", "width_rules")

In [10]:
df.columns

['id',
 'bbox',
 'version',
 'sources',
 'geometry',
 'subtype',
 'class',
 'names',
 'connectors',
 'routes',
 'subclass',
 'level_rules',
 'road_surface',
 'road_flags',
 'width_rules']

# Sedona DataFrame to GeoPandas

In [11]:
## Inefficient way of converting spatial DataFrame to GeoPandas

In [12]:
import geopandas as gpd

start = time()
gdf = gpd.GeoDataFrame(df.toPandas(), geometry="geometry")
print(f"converted in {time() - start}")

converted in 21.673442602157593


In [13]:
## using the GeoArrow conversion

In [14]:
start = time()
gdf = gpd.GeoDataFrame.from_arrow(dataframe_to_arrow(df))
print(f"converted in {time() - start}")

converted in 8.708709239959717


# Creating Sedona DataFrame from shapely objects

In [15]:
from shapely.geometry import Point
import sedona.spark.sql.types as st
import pyspark.sql.types as t
 
schema = t.StructType(
    [
        t.StructField("id", t.IntegerType()),
        t.StructField("geom", st.GeometryType()),
    ]
)
 
shapely_df = sedona.createDataFrame([
    [1, Point(21, 52)],
    [2, Point(21, 45)]
], schema=schema)

In [16]:
shapely_df.show()

+---+-------------+
| id|         geom|
+---+-------------+
|  1|POINT (21 52)|
|  2|POINT (21 45)|
+---+-------------+



In [17]:
sedona.createDataFrame([
    {"id": 1, "geom": Point(21, 52)},
    {"id": 2, "geom": Point(21, 45)}
]).show()

+-------------+---+
|         geom| id|
+-------------+---+
|POINT (21 52)|  1|
|POINT (21 45)|  2|
+-------------+---+



# Sedona DataFrame from GeoPandas

In [18]:
gdf_subset = gdf[["id", "bbox", "version", "subtype", "class", "geometry", "names"]]

In [19]:
start = time()
sedona.createDataFrame(gdf_subset)
print(f"converted in {time() - start}")

converted in 2.017025947570801


In [20]:
# Sedona DataFrame from geopandas using Apache Arrow

In [21]:
start = time()
create_spatial_dataframe(sedona, gdf_subset)
print(f"converted in {time() - start}")

converted in 0.8856635093688965


# Writing own UDF function

In [31]:
import pyspark.sql.functions as f
import sedona.spark.sql.types as st
import shapely.geometry.base as b
 
def create_buffer_distance(
    s: b.BaseGeometry,
    distance_from: float,
    distance_to: float
) -> b.BaseGeometry:
    buffer_a = s.buffer(distance_from)
    buffer_b = s.buffer(distance_to)
    return buffer_b.difference(buffer_a)
 
buffer_distanced_udf = f.udf(create_buffer_distance, st.GeometryType())
 
sedona.udf.register(
    "ST_BufferDistanceNonVectorized",
    buffer_distanced_udf
)

In [32]:
df.createOrReplaceTempView("roads")

In [33]:
sedona.sql(
"""
    SELECT 
        ST_BufferDistanceNonVectorized(geometry, CAST(0.0001 AS FLOAT), CAST(0.0002 AS FLOAT)) AS geometry
    FROM roads
    """
).show()

[Stage 18:>                                                         (0 + 1) / 1]

+--------------------+
|            geometry|
+--------------------+
|POLYGON ((3.70448...|
|POLYGON ((5.33776...|
|POLYGON ((2.18222...|
|POLYGON ((3.70527...|
|POLYGON ((3.06424...|
|POLYGON ((8.45250...|
|POLYGON ((-0.6447...|
|POLYGON ((8.91493...|
|POLYGON ((1.44936...|
|POLYGON ((2.63515...|
|POLYGON ((2.08226...|
|POLYGON ((2.08236...|
|POLYGON ((2.08817...|
|POLYGON ((2.08926...|
|POLYGON ((2.08997...|
|POLYGON ((2.09033...|
|POLYGON ((2.09064...|
|POLYGON ((2.09172...|
|POLYGON ((2.09142...|
|POLYGON ((2.09272...|
+--------------------+
only showing top 20 rows



# Writing Vectorized UDF (better performance)

In [20]:
from sedona.spark.sql.functions import sedona_vectorized_udf
from sedona.spark.sql.types import GeometryType

@sedona_vectorized_udf(return_type=GeometryType())
def vectorized_symmetrical_buffer_distance_udf(
        geom: b.BaseGeometry
) -> b.BaseGeometry:
    return create_buffer_distance(geom, 0.0001, 0.0002)

In [21]:
df.select(
    vectorized_symmetrical_buffer_distance_udf(f.col("geometry"))
).show(10)

[Stage 14:>                                                         (0 + 1) / 1]

+------------------------------+
|SedonaPandasArrowUDF(geometry)|
+------------------------------+
|          POLYGON ((3.70448...|
|          POLYGON ((5.33776...|
|          POLYGON ((2.18222...|
|          POLYGON ((3.70527...|
|          POLYGON ((3.06424...|
|          POLYGON ((8.45250...|
|          POLYGON ((-0.6447...|
|          POLYGON ((8.91493...|
|          POLYGON ((1.44936...|
|          POLYGON ((2.63515...|
+------------------------------+
only showing top 10 rows

